In [ ]:
import pandas as pd
df = pd.read_csv("loan_filtered.csv")
df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# 贷款金额分布的直方图
plt.figure(figsize=(10, 6))
sns.histplot(df['loan_amnt'], bins=50, kde=True)
plt.title('Loan Amount Distribution')
plt.xlabel('Loan Amount')
plt.ylabel('Frequency')
plt.show()

In [ ]:
category_counts = df['loan_status'].value_counts()

# 创建横向条形图
category_counts.plot(kind='barh')

# 设置图表标题和坐标轴标签
plt.title('Loan Status Distribution')
plt.xlabel('Counts')
plt.ylabel('Categories')

plt.show()

In [ ]:
# ==================== 特征选择（Feature Selection） ====================
# 以下步骤从《1 数据初步清洗》notebook 移入，属于特征工程范畴：
# 1. 删除高基数列（id、url、emp_title 等标识符/近唯一列）
# 2. 删除类别严重不平衡列
# 3. 删除与信用评估无关或冗余的列
# 输入：loan_filtered.csv（52 列，仅完成缺失值处理）
# 输出：进入编码阶段的 df（40 列）
print(f"读入数据维度: {df.shape}")

In [ ]:
# 特征选择1：查看各非数值列的唯一值数量，识别高基数列
# （id、member_id、url、emp_title、title、zip_code 等唯一值极多，不具备预测能力）
non_float64_columns = df.select_dtypes(exclude=["float64"])
unique_counts = non_float64_columns.nunique()
print(unique_counts)

In [ ]:
# 删除唯一值 > 697 的高基数列
columns_to_drop = unique_counts[unique_counts > 697].index
df = df.drop(columns=columns_to_drop)
print(f"删除高基数列后维度: {df.shape}")
df.head()

In [ ]:
# 特征选择2：检查类别严重不平衡的列
print("pymnt_plan 分布:")
print(df.pymnt_plan.value_counts())
print()
print("application_type 分布:")
print(df.application_type.value_counts())

In [ ]:
# 特征选择3：删除冗余、不平衡、与信用评估无关的列
# - application_type、pymnt_plan：类别极度不平衡（>99.9% 为同一类别）
# - last_pymnt_amnt、last_pymnt_d：还款结果信息，与信用评估存在因果倒置
# - sub_grade：与 grade 高度冗余，保留 grade 即可
# - policy_code：只有唯一一个取值，不含任何信息
cols_to_drop = ["application_type", "pymnt_plan", "last_pymnt_amnt",
                "sub_grade", "policy_code", "last_pymnt_d"]
df = df.drop(columns=cols_to_drop)
print(f"特征选择完成后维度: {df.shape}")

In [ ]:
# Data encoding1：将某些有顺序的类别变量重定义为定序变量
df['grade'] =df['grade'].replace({'A':1,'B':2,'C':3,'D':4,'E':5,'F':6,'G':7})
print(df.grade.value_counts())

df["home_ownership"] = df["home_ownership"].replace({"MORTGAGE":2,"RENT":3,"OWN":1,"OTHER":4,"NONE":5,"ANY":6})
print(df.home_ownership.value_counts())

df['verification_status']=df['verification_status'].replace({'Source Verified': 1,'Verified': 2,'Not Verified': 3,})
print(df.verification_status.value_counts())

df['loan_status'] = df['loan_status'].replace({'Fully Paid':1,
'Current':2,
'In Grace Period':3,
'Issued':4,
'Late (16-30 days)':5,
'Does not meet the credit policy. Status:Fully Paid':6,
'Late (31-120 days)':7,
'Default':8,
'Does not meet the credit policy. Status:Charged Off':9,
'Charged Off':10})
print(df.loan_status.value_counts())

df['initial_list_status']=df['initial_list_status'].replace({'f': 0,'w': 1})
print(df.initial_list_status.value_counts())

In [ ]:
# 对数值+字符串的变量做一些处理
df['term'] = df['term'].str.split().str[0].astype(int)
df.term.value_counts()

In [ ]:
df['emp_length'] = df['emp_length'].replace('< 1 year', '0.5 year')
df['emp_length'] = df['emp_length'].replace('10+ years', '10 year')
df['emp_length'] = df['emp_length'].str.split().str[0].astype(float)
df['emp_length'] = df['emp_length']
print(df.emp_length.value_counts())

In [ ]:
# 将日期变量转化为数值变量：与最近的日期之间的距离
df['issue_d'] = pd.to_datetime(df['issue_d'])
df['last_credit_pull_d'] = pd.to_datetime(df['last_credit_pull_d'])
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])

# 找到参照日期（最近的日期）
reference_date = df['issue_d'].max()
df['issue_d'] = (reference_date.year - df['issue_d'].dt.year) * 12 + (reference_date.month - df['issue_d'].dt.month)
# 找到参照日期（最近的日期）
reference_date = df['last_credit_pull_d'].max()
df['last_credit_pull_d'] = (reference_date.year - df['last_credit_pull_d'].dt.year) * 12 + (reference_date.month - df['last_credit_pull_d'].dt.month)
# 找到参照日期（最近的日期）
reference_date = df['earliest_cr_line'].max()
df['earliest_cr_line'] = (reference_date.year - df['earliest_cr_line'].dt.year) * 12 + (reference_date.month - df['earliest_cr_line'].dt.month)

In [ ]:
# 无法定序的类别变量用标签编码做处理（考虑到独热编码可能使模型解释性更强，但新增维数太多）
from sklearn.preprocessing import LabelEncoder

# 创建LabelEncoder的实例
label_encoder = LabelEncoder()

# 进行标签编码
df['purpose_'] = label_encoder.fit_transform(df['purpose'])
df['addr_state_'] = label_encoder.fit_transform(df['addr_state'])

df2 = df.drop(['purpose','addr_state'], axis=1)

# 查看转换后的数据
df[['purpose', 'purpose_', 'addr_state', 'addr_state_']]

In [ ]:
df2.head(10)

In [ ]:
df2.corr()

In [ ]:
# 把与grade的相关性过高的列int_rate删去(事实上因果倒置)
df3 = df2.drop('int_rate', axis=1)

# 降维：由上面的相关矩阵可见loan_amnt、funded_amnt、funded_amnt_inv、installment有较强相关性，考虑只保留loan_amnt.
df3 = df3.drop(['funded_amnt_inv', 'installment', 'funded_amnt'], axis=1)
df3.head(10)

In [ ]:
# 将数据框写入新CSV文件
df3.to_csv('loan_encoded.csv', index=False)